# COMPASS — training & evaluation on Kaggle (GPU)

Thin runner: **all logic lives in the git repo**, this notebook only executes it
(`git pull` delivers code updates; never edit code here).

Before running:
- **Accelerator: GPU** (T4 works; ~16 GB)
- **Internet: ON** — needed once to download ImageNet weights (~85 MB).
  If OFF, training still runs but from random init (a warning is printed).
- The `compass-preprocessed` dataset is attached as input data
  (`/kaggle/input/datasets/muhammadumerbsse/compass-preprocessed`).

Stage 1 = 150 epochs, AdamW 1e-4, `L_seg + 0.5·L_contra + 0.3·L_cps`.
The cell auto-**resumes** from `ckpt_last.pt` if the session was interrupted —
just re-run it.

In [ ]:
import os, subprocess

REPO = "https://github.com/Umer-3085/COMPASS.git"
DIR = "/kaggle/working/COMPASS"
if not os.path.exists(DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO, DIR], check=True)
else:
    subprocess.run(["git", "pull"], cwd=DIR, check=True)
os.chdir(DIR)
print("repo ready at", os.getcwd())

In [ ]:
import torch, os
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU!)")

DATA = "/kaggle/input/datasets/muhammadumerbsse/compass-preprocessed"
print("data:", sorted(os.listdir(DATA)))
assert os.path.exists(os.path.join(DATA, "template", "ct_template.nii.gz"))

In [ ]:
# Stage 1: 150 epochs. Re-run this cell to resume after interruption.
ckpt = "/kaggle/working/runs/stage1/ckpt_last.pt"
resume = ["--resume"] if os.path.exists(ckpt) else []
subprocess.run([
    "python", "-m", "src.train", "--stage", "1",
    "--data-root", DATA,
    "--out", "/kaggle/working/runs/stage1",
    "--batch-size", "8", "--num-workers", "2", "--amp",
    "--log-every", "50", *resume,
], check=True)

**Notes**
- `runs/stage1/train_log.csv` grows one row per epoch; `ckpt_best.pt` tracks
  the best validation mean foreground Dice.
- `/kaggle/working` is wiped when the session ends → after Stage 1 finishes,
  run the download cell at the bottom (or copy checkpoints into a Kaggle
  dataset) before closing.
- Stage 2 (frozen encoders+fusion, new decoder + evidential head,
  75 epochs, AdamW 5e-5) needs the Stage-1 best checkpoint in the same session.

In [ ]:
# Stage 2 (run after Stage 1 completes in this session)
s1_best = "/kaggle/working/runs/stage1/ckpt_best.pt"
assert os.path.exists(s1_best), "finish Stage 1 first"
ckpt2 = "/kaggle/working/runs/stage2/ckpt_last.pt"
resume2 = ["--resume"] if os.path.exists(ckpt2) else []
subprocess.run([
    "python", "-m", "src.train", "--stage", "2",
    "--init-from", s1_best,
    "--data-root", DATA,
    "--out", "/kaggle/working/runs/stage2",
    "--batch-size", "8", "--num-workers", "2", "--amp",
    "--log-every", "50", *resume2,
], check=True)

In [ ]:
# Six-condition evaluation (uses the Stage-2 best checkpoint)
s2_best = "/kaggle/working/runs/stage2/ckpt_best.pt"
assert os.path.exists(s2_best), "finish Stage 2 first"
subprocess.run([
    "python", "-m", "src.evaluate",
    "--ckpt", s2_best,
    "--data-root", DATA,
    "--out", "/kaggle/working/runs/eval",
    "--batch-size", "8", "--num-workers", "2",
], check=True)

In [ ]:
# Pack + download logs and evaluation tables (checkpoints stay local-only)
import shutil
from IPython.display import FileLink

shutil.make_archive("/kaggle/working/compass_results", "zip",
                    "/kaggle/working/runs")
FileLink("/kaggle/working/compass_results.zip")